# DB하이텍(000990) 2025년 RAG 인덱스 구축 — 파일럿

**범위:** DB하이텍 1개사, 2025년 3종 보고서(사업/지배구조/지속가능경영)만 대상으로 한 파일럿 실행.
5개사 확장 전, 파이프라인이 실제로 동작하는지 검증하기 위한 목적.

**구성:**
- 🟢 팀 공용 코드(`build_rag_index.py`)에서 **그대로 가져온 부분** — 표/텍스트 추출, 섹션 탐지,
  사업보고서·지배구조보고서 처리, BM25 하이브리드 리트리버
- 🟡 **내가 추가한 부분** — 지속가능경영보고서 전용 처리(하이브리드 파싱 결과 연동, 잡음 제거),
  팀 리뷰용 CSV 내보내기

각 코드 셀 위 마크다운에 어느 쪽인지 표시해뒀습니다.

## 1. 설치

In [1]:
!pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
    chromadb pdfplumber kiwipiepy rank_bm25 tiktoken pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 617.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━

## 2. Google Drive 마운트 + API 키

지속가능경영보고서는 로컬(hybrid_parse.py)에서 LlamaParse로 먼저 재처리한 결과(`*_cleaned.jsonl`)를
써야 합니다. 이 파일을 Drive의 아래 경로에 미리 업로드해두세요 (본인 개인 작업 폴더 기준):

```
ESG 팀프로젝트/2. ESG 채점 세트/인덱스 생성 코드/RAG_진혜원/parsed_hybrid/000990_2025_sustainability_report_cleaned.jsonl
```


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API Key 입력: ")

Mounted at /content/drive
Google AI Studio API Key 입력: ··········


## 3. 설정값 (CONFIG)

🟢 하이퍼파라미터는 팀 확정값 그대로. 경로만 이번 파일럿(DB하이텍 2025) 대상으로 한정.

In [3]:
DRIVE_ROOT = "/content/drive/MyDrive/ESG 팀프로젝트"

CONFIG = {
    # --- 팀 확정값 (그대로) ---
    "embedding_model": "gemini-embedding-001",
    "chunk_size_tokens": 800,
    "chunk_overlap_tokens": 100,
    "bm25_tokenizer": "kiwi",
    "retrieval_k": 10,
    "hybrid_weights": [0.5, 0.5],  # 임시값, 평가셋 튜닝 전까지 유지

    # --- 이번 파일럿 전용 경로 ---
    "persist_dir": f"{DRIVE_ROOT}/2. ESG 채점 세트/인덱스 생성 코드/RAG_진혜원/chroma_db",
    "checkpoint_log": f"{DRIVE_ROOT}/2. ESG 채점 세트/인덱스 생성 코드/RAG_진혜원/checkpoint_completed.json",
    "reports_root": f"{DRIVE_ROOT}/2. ESG 채점 세트/보고서 3종/DB하이텍(000990)",
    # 🟡 내가 추가: hybrid_parse.py + clean_hybrid_chunks.py 결과 (LlamaParse 재처리 + 잡음 제거 완료본)
    "hybrid_sustainability_jsonl": f"{DRIVE_ROOT}/2. ESG 채점 세트/인덱스 생성 코드/RAG_진혜원/parsed_hybrid/000990_2025_sustainability_report_cleaned.jsonl",
}

TARGET_DOCS = [
    {"company_code": "000990", "company_name": "DB하이텍", "year": 2025,
     "doc_type": "business_report",
     "file_path": f'{CONFIG["reports_root"]}/000990_2025_business.pdf'},
    {"company_code": "000990", "company_name": "DB하이텍", "year": 2025,
     "doc_type": "governance_report",
     "file_path": f'{CONFIG["reports_root"]}/000990_2025_governance.pdf'},
    # sustainability_report는 파일 경로 대신 hybrid jsonl에서 로드 (아래 6번 참고)
]

## 4. 라이브러리 import + 토크나이저 설정

🟢 팀 공용 코드 그대로.

In [4]:
import re
import json
import hashlib
from datetime import datetime
from pathlib import Path

import pdfplumber
import tiktoken
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

kiwi = Kiwi()
RRF_K = 60  # Reciprocal Rank Fusion 표준 상수

try:
    _enc = tiktoken.encoding_for_model(CONFIG["embedding_model"])
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")

def token_len(text: str) -> int:
    return len(_enc.encode(text))

def kiwi_tokenize(text: str):
    '''BM25용 한국어 형태소 분석 토크나이저 (조사 분리 목적).'''
    return [t.form for t in kiwi.tokenize(text)]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size_tokens"],
    chunk_overlap=CONFIG["chunk_overlap_tokens"],
    length_function=token_len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

/tmp/ipykernel_3091/4209643713.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


## 5. 표/텍스트 추출 유틸 + 메타데이터 스키마

🟢 팀 공용 코드 그대로. 사업보고서·지배구조보고서 처리에 사용 (표-텍스트 bbox 분리, 표 토큰 단위 분할).

In [5]:
def clean_cell(cell) -> str:
    """표 셀 안의 줄바꿈/파이프가 마크다운 표 구조를 깨는 것 방지."""
    if cell is None:
        return ""
    return str(cell).replace("\n", " ").replace("|", "\\|").strip()


def chunk_table(table_data: list, max_tokens: int) -> list:
    """큰 표를 token 기준으로 행 단위 분할 (각 조각에 헤더 반복)."""
    header = [clean_cell(c) for c in table_data[0]]
    data_rows = table_data[1:]
    header_md = "| " + " | ".join(header) + " |\n" + "|" + "|".join(["---"] * len(header)) + "|"

    def render(rows):
        body = "\n".join("| " + " | ".join(clean_cell(c) for c in r) + " |" for r in rows)
        return header_md + ("\n" + body if body else "")

    chunks, current_rows = [], []
    for row in data_rows:
        candidate = current_rows + [row]
        if token_len(render(candidate)) > max_tokens and current_rows:
            chunks.append({"markdown": render(current_rows), "rows": current_rows})
            current_rows = [row]
        else:
            current_rows = candidate
    if current_rows:
        chunks.append({"markdown": render(current_rows), "rows": current_rows})
    return chunks or [{"markdown": header_md, "rows": []}]


def extract_page_content(pdf_page):
    """표 영역 bbox를 찾고, 그걸 제외한 텍스트만 반환 (표-텍스트 중복 방지)."""
    tables = pdf_page.find_tables()
    table_bboxes = [t.bbox for t in tables]

    table_data_list = []
    for t in tables:
        try:
            extracted = t.extract()
        except Exception:
            continue
        if extracted and len(extracted) >= 2:
            table_data_list.append(extracted)

    def not_in_table(obj):
        x0, top, x1, bottom = obj.get("x0"), obj.get("top"), obj.get("x1"), obj.get("bottom")
        if x0 is None:
            return True
        for (tx0, ttop, tx1, tbottom) in table_bboxes:
            if x0 >= tx0 - 1 and x1 <= tx1 + 1 and top >= ttop - 1 and bottom <= tbottom + 1:
                return False
        return True

    non_table_page = pdf_page.filter(not_in_table)
    text = non_table_page.extract_text() or ""
    return text, table_data_list


def base_metadata(company_code, company_name, year, doc_type, doc_id, chunk_id, page,
                   content_type, source_file_hash, doc_indexed_at, table_rows=None, section_title=""):
    """Chroma는 메타데이터로 str/int/float/bool만 허용 -> None은 빈 문자열, list는 JSON 문자열로."""
    return {
        "company_code": company_code, "company_name": company_name, "year": year,
        "doc_type": doc_type, "doc_id": doc_id, "chunk_id": chunk_id, "page": page,
        "section_title": section_title, "content_type": content_type,
        "table_data": json.dumps(table_rows, ensure_ascii=False) if table_rows else "",
        "esg_area": "", "applicable_item_codes": json.dumps([], ensure_ascii=False),
        "embedding_model": CONFIG["embedding_model"], "indexed_at": doc_indexed_at,
        "source_file_hash": source_file_hash,
    }

## 6. 섹션 제목 키워드 탐지 + 문서 처리

🟢 팀 공용 코드 그대로. 사업보고서·지배구조보고서만 섹션 키워드가 정의되어 있음
(지속가능경영보고서는 목차 규칙이 없어 팀 결정대로 빈 문자열 유지).

In [6]:
SECTION_KEYWORDS = {
    "business_report": [
        "회사의 개요", "사업의 내용", "재무에 관한 사항", "이사의 경영진단",
        "회계감사인의 감사의견", "이사회 등 회사의 기관에 관한 사항",
        "주주에 관한 사항", "임원 및 직원 등에 관한 사항",
        "계열회사 등에 관한 사항", "이해관계자와의 거래내용",
    ],
    "governance_report": [
        "주주", "이사회", "감사기구", "이사 감사의 보수",
        "내부통제", "배당", "주주총회", "공시",
    ],
}

def _normalize(line: str) -> str:
    return line.replace("·", "").replace(" ", "").replace("/", "")

def detect_section_title(line: str, doc_type: str):
    keywords = SECTION_KEYWORDS.get(doc_type)
    if not keywords:
        return None
    stripped = line.strip()
    if not stripped or len(stripped) > 40:
        return None
    norm_line = _normalize(stripped)
    for kw in keywords:
        if _normalize(kw) in norm_line:
            return kw
    return None


def process_document(company_code, company_name, year, doc_type, file_path,
                      source_file_hash, doc_indexed_at):
    """PDF 1개 -> Document 리스트. 사업보고서/지배구조보고서 전용
    (지속가능경영보고서는 아래 7번의 load_hybrid_sustainability_chunks() 사용)."""
    doc_id = f"{company_code}_{year}_{doc_type}"
    chunks, counter, current_section = [], 0, ""

    with pdfplumber.open(file_path) as pdf:
        for page_number, page in enumerate(pdf.pages):
            text, tables = extract_page_content(page)

            for line in text.split("\n"):
                detected = detect_section_title(line, doc_type)
                if detected:
                    current_section = detected

            for table_data in tables:
                for piece in chunk_table(table_data, CONFIG["chunk_size_tokens"]):
                    counter += 1
                    chunk_id = f"{doc_id}_{counter:04d}"
                    meta = base_metadata(company_code, company_name, year, doc_type, doc_id,
                                          chunk_id, page_number + 1, "table", source_file_hash,
                                          doc_indexed_at, table_rows=piece["rows"],
                                          section_title=current_section)
                    meta["chunk_char_count"] = len(piece["markdown"])
                    chunks.append(Document(page_content=piece["markdown"], metadata=meta))

            if not text.strip():
                continue
            for chunk_text_piece in splitter.split_text(text):
                counter += 1
                chunk_id = f"{doc_id}_{counter:04d}"
                meta = base_metadata(company_code, company_name, year, doc_type, doc_id,
                                      chunk_id, page_number + 1, "text", source_file_hash,
                                      doc_indexed_at, section_title=current_section)
                meta["chunk_char_count"] = len(chunk_text_piece)
                chunks.append(Document(page_content=chunk_text_piece, metadata=meta))

    return chunks

## 7. 🟡 지속가능경영보고서 전용 처리 (내가 추가한 부분)

**왜 필요한가:** 이 문서는 본문이 폰트가 아니라 벡터 윤곽선(curve)으로 변환되어 있어서
`process_document()`(pdfplumber 기반)로 처리하면 텍스트가 거의 안 뽑힘
(실측: 92페이지 중 87페이지, 95% 텍스트 손실 확인).

**어떻게 해결했나:**
1. 별도 스크립트 `hybrid_parse.py`로 PyMuPDF 1차 추출 → 텍스트/표가 부족한 페이지만
   골라서 LlamaParse(Vision-LLM)로 재처리 → JSONL로 저장 (전체 페이지를 다 보내지 않아
   API 비용 절감)
2. LlamaParse 결과에 사이드바 탭 라벨("ESG Performance", "Company Profile" 등)과
   러닝헤더(보고서 제목)가 본문과 이어붙어 나오는 경우가 있어, `clean_hybrid_chunks.py`로
   후처리 필터링 (표는 파이프 문자를 구조로 쓰므로 절대 건드리지 않음)
3. 이 노트북에서는 그 최종 결과(`*_cleaned.jsonl`)를 읽어서 팀의 `base_metadata()` 스키마에
   맞는 Document로 변환만 함 (아래 함수)

In [7]:
def load_hybrid_sustainability_chunks(jsonl_path: str, company_code: str,
                                       company_name: str, year: int, doc_indexed_at: str,
                                       original_pdf_path: str):
    """hybrid_parse.py + clean_hybrid_chunks.py 결과(JSONL)를 팀 스키마 Document로 변환."""
    path = Path(jsonl_path)
    if not path.exists():
        raise FileNotFoundError(
            f"hybrid_parse.py 결과 파일이 없습니다: {jsonl_path}\n"
            "먼저 로컬에서 hybrid_parse.py -> clean_hybrid_chunks.py를 실행해 Drive에 올려두세요."
        )

    doc_type = "sustainability_report"
    doc_id = f"{company_code}_{year}_{doc_type}"

    pdf_path = Path(original_pdf_path)
    if pdf_path.exists():
        with open(pdf_path, "rb") as f:
            source_file_hash = hashlib.md5(f.read()).hexdigest()
    else:
        source_file_hash = "unknown"

    documents = []
    with open(path, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            row = json.loads(line)
            content = row["content"]
            if not content.strip():
                continue
            chunk_id = f"{doc_id}_{idx:04d}"
            meta = base_metadata(company_code, company_name, year, doc_type, doc_id,
                                  chunk_id, row.get("page", 0), row.get("content_type", "text"),
                                  source_file_hash, doc_indexed_at,
                                  table_rows=row.get("table_data"),
                                  section_title="")  # 목차 규칙 없어 팀 결정대로 빈 문자열
            meta["chunk_char_count"] = len(content)
            documents.append(Document(page_content=content, metadata=meta))

    return documents

## 8. 임베딩 + 체크포인트 인덱싱

🟢 구조는 팀 공용 코드(Cell 7)와 동일 — 파일 해시 기반 체크포인트로 변경분만 재인덱싱.
🟡 대상만 이번 파일럿(DB하이텍 2025 3종)으로 한정, sustainability는 7번 로더 사용.

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)

if Path(CONFIG["checkpoint_log"]).exists():
    with open(CONFIG["checkpoint_log"], encoding="utf-8") as f:
        checkpoint = json.load(f)
else:
    checkpoint = {}

all_chunks_for_csv = []  # 🟡 내가 추가: 10번 CSV 내보내기용 누적 리스트

# --- business_report, governance_report ---
for target in TARGET_DOCS:
    doc_id = f"{target['company_code']}_{target['year']}_{target['doc_type']}"
    file_path = target["file_path"]

    if not Path(file_path).exists():
        print(f"[건너뜀] 파일 없음: {file_path}")
        continue

    with open(file_path, "rb") as f:
        current_hash = hashlib.md5(f.read()).hexdigest()

    prior = checkpoint.get(doc_id)
    if prior and prior["source_file_hash"] == current_hash:
        print(f"스킵 (변경 없음): {doc_id}")
        continue
    if prior:
        print(f"PDF 변경 감지 -> 재인덱싱: {doc_id}")
        vectorstore.delete(where={"doc_id": doc_id})

    print(f"처리 중: {doc_id}")
    doc_indexed_at = datetime.now().isoformat()
    chunks = process_document(
        company_code=target["company_code"], company_name=target["company_name"],
        year=target["year"], doc_type=target["doc_type"], file_path=file_path,
        source_file_hash=current_hash, doc_indexed_at=doc_indexed_at,
    )
    if chunks:
        vectorstore.add_documents(chunks, ids=[c.metadata["chunk_id"] for c in chunks])
        print(f"  -> {len(chunks)}개 청크 적재")
        all_chunks_for_csv.extend(chunks)

    checkpoint[doc_id] = {"source_file_hash": current_hash, "indexed_at": doc_indexed_at}
    with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)

# --- sustainability_report (🟡 hybrid 결과 사용) ---
sus_doc_id = "000990_2025_sustainability_report"
hybrid_path = CONFIG["hybrid_sustainability_jsonl"]
sus_pdf_path = f'{CONFIG["reports_root"]}/000990_2025_sustainability.pdf'

if Path(hybrid_path).exists():
    with open(hybrid_path, "rb") as f:
        current_hash = hashlib.md5(f.read()).hexdigest()

    prior = checkpoint.get(sus_doc_id)
    if prior and prior["source_file_hash"] == current_hash:
        print(f"스킵 (변경 없음): {sus_doc_id}")
    else:
        if prior:
            print(f"결과 변경 감지 -> 재인덱싱: {sus_doc_id}")
            vectorstore.delete(where={"doc_id": sus_doc_id})

        print(f"처리 중: {sus_doc_id} (hybrid_parse.py 결과 사용)")
        doc_indexed_at = datetime.now().isoformat()
        chunks = load_hybrid_sustainability_chunks(
            hybrid_path, "000990", "DB하이텍", 2025, doc_indexed_at, sus_pdf_path,
        )
        if chunks:
            vectorstore.add_documents(chunks, ids=[c.metadata["chunk_id"] for c in chunks])
            print(f"  -> {len(chunks)}개 청크 적재")
            all_chunks_for_csv.extend(chunks)

        checkpoint[sus_doc_id] = {"source_file_hash": current_hash, "indexed_at": doc_indexed_at}
        with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
else:
    print(f"[건너뜀] hybrid 결과 없음: {hybrid_path}")

print(f"\n=== 인덱싱 완료: {len(checkpoint)}건 처리됨 ===")

처리 중: 000990_2025_business_report
  -> 1105개 청크 적재


처리 중: 000990_2025_governance_report


  -> 169개 청크 적재
처리 중: 000990_2025_sustainability_report (hybrid_parse.py 결과 사용)
  -> 275개 청크 적재

=== 인덱싱 완료: 3건 처리됨 ===


## 9. 필터링 지원 하이브리드 리트리버

🟢 팀 공용 코드 그대로. BM25를 전체 코퍼스가 아니라 필터(company_code 등)로 좁혀진
부분집합에서만 만들어서, 다른 회사로 확장해도 BM25 재구축 비용이 낮음.

In [9]:
class FilteredHybridRetriever:
    def __init__(self, vectorstore, tokenize_func, k=10, weights=(0.5, 0.5)):
        self.vectorstore = vectorstore
        self.tokenize_func = tokenize_func
        self.k = k
        self.weights = weights
        self._bm25_cache = {}

    def _get_bm25_retriever(self, filter_dict):
        sig = tuple(sorted((filter_dict or {}).items()))
        if sig in self._bm25_cache:
            return self._bm25_cache[sig]
        raw = self.vectorstore.get(where=filter_dict if filter_dict else None,
                                    include=["documents", "metadatas"])
        docs = [Document(page_content=d, metadata=m) for d, m in zip(raw["documents"], raw["metadatas"])]
        if not docs:
            self._bm25_cache[sig] = None
            return None
        retriever = BM25Retriever.from_documents(docs, preprocess_func=self.tokenize_func)
        retriever.k = self.k
        self._bm25_cache[sig] = retriever
        return retriever

    def invoke(self, query, filter=None, top_n=None):
        top_n = top_n or self.k
        dense_docs = self.vectorstore.similarity_search(query, k=self.k, filter=filter)
        bm25_retriever = self._get_bm25_retriever(filter)
        bm25_docs = bm25_retriever.invoke(query) if bm25_retriever else []

        scores, lookup = {}, {}
        for rank, doc in enumerate(dense_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[0] * (1.0 / (RRF_K + rank))
            lookup[key] = doc
        for rank, doc in enumerate(bm25_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[1] * (1.0 / (RRF_K + rank))
            lookup[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [lookup[k] for k, _ in ranked[:top_n]]

hybrid_retriever = FilteredHybridRetriever(
    vectorstore=vectorstore, tokenize_func=kiwi_tokenize,
    k=CONFIG["retrieval_k"], weights=CONFIG["hybrid_weights"],
)

## 10. 동작 확인 테스트

🟢 팀 코드 패턴(필터 검증 + 하이브리드 검색) 재사용. 질의만 이번 파일럿 문서에 맞춰 작성.

In [10]:
test_filtered = vectorstore.similarity_search("사외이사 비율", k=5, filter={"company_code": "000990"})
for d in test_filtered:
    assert d.metadata["company_code"] == "000990", "필터링 실패 -- 다른 회사 청크 혼입"
print(f"Dense 필터 테스트 통과: {len(test_filtered)}건")

hybrid_results = hybrid_retriever.invoke("이사회 산하 위원회 구성 현황", filter={"company_code": "000990"})
for d in hybrid_results:
    assert d.metadata["company_code"] == "000990", "하이브리드 필터링 실패"
print(f"하이브리드 필터 테스트 통과: {len(hybrid_results)}건\n")

for q in ["온실가스 배출량 저감 목표", "협력사 안전보건 지원 프로그램", "이사회 독립성 확보 방안"]:
    results = hybrid_retriever.invoke(q, filter={"company_code": "000990"}, top_n=3)
    print(f"[질문] {q}")
    for r in results:
        print(f"  - {r.metadata['doc_type']} p.{r.metadata['page']}: {r.page_content[:60]}")
    print()

Dense 필터 테스트 통과: 5건
하이브리드 필터 테스트 통과: 10건

[질문] 온실가스 배출량 저감 목표
  - sustainability_report p.21: 너지 저감 예상량 산출 | 5 모니터링 및 검증 목표수정 

## ▶ 온실가스 배출량

DB하이텍의 온실가스
  - sustainability_report p.21: # 온실가스 저감

DB하이텍은 지구에 미치는 악영향을 최소화 하기 위해 온실가스 배출 최소화를 위해 노력하
  - sustainability_report p.20: # 온실가스 저감

## ▶ 온실가스 저감 목표

DB하이텍은 기후 변화에 따른 지속가능한 성장에 동참하기 

[질문] 협력사 안전보건 지원 프로그램
  - sustainability_report p.46: ESG DATA 협력사ESH 관리DB하이텍은협력업체와의공생협의체를통한안전보건관리체계를구축하고자율관리능력향상이
  - sustainability_report p.47: ESG DATA 협력사ESH 관리DB하이텍은매월협력사정기협의체회의를통해주요안전보건관리현황과관련법규제·개정에관
  - sustainability_report p.51: ESG DATA 동반성장지원프로그램DB하이텍은협력사와의동반성장을위해다양한지원프로그램을운영하고있으며,이를통해국

[질문] 이사회 독립성 확보 방안
  - governance_report p.29: 현재 당사의 규모와 사업현황을 고려할 때 업무집행 기능과 감독기능을 분리하는 것은 아직 시기상조라고 판단하고
  - governance_report p.4: 위원회를 운영하고 있으며, 이사회는 사안에 따라 결의에 대한 권한을 각 위원회에 위임하여 이사회 운영의 효율
  - governance_report p.25: 수 있는 투명한 절차를 통하여 선임되어야 한다.
[304100] (세부원칙 4-1) - 이사회는 회사의 지속



## 11. 🟡 팀 리뷰용 CSV 내보내기 (내가 추가한 부분)

코드를 실행하지 않고도 엑셀/시트에서 청크 결과를 훑어볼 수 있도록 요약 CSV를 만듭니다.
전체 본문(`content`)은 너무 길어서 앞 100자만 미리보기로 넣고, 전체 글자수는 별도 컬럼으로 둠.

In [11]:
import pandas as pd

rows = []
for c in all_chunks_for_csv:
    m = c.metadata
    rows.append({
        "chunk_id": m["chunk_id"],
        "doc_type": m["doc_type"],
        "page": m["page"],
        "section_title": m["section_title"],
        "content_type": m["content_type"],
        "chunk_char_count": m["chunk_char_count"],
        "content_preview": c.page_content[:100].replace("\n", " "),
    })

df = pd.DataFrame(rows)
csv_path = f'{CONFIG["persist_dir"]}/../chunks_summary.csv'
df.to_csv(csv_path, index=False, encoding="utf-8-sig")  # 엑셀 한글 깨짐 방지
print(f"CSV 저장 완료: {csv_path} ({len(df)}행)")
df.head(10)

CSV 저장 완료: /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 생성 코드/RAG_진혜원/chroma_db/../chunks_summary.csv (1549행)


,chunk_id,doc_type,page,section_title,content_type,chunk_char_count,content_preview
0,000990_2025_business_report_0001,business_report,1,,text,5211,목 차 사 업 보 고 서....................................
1,000990_2025_business_report_0002,business_report,1,,text,1143,12. 관계기업투자 (연결)..................................
2,000990_2025_business_report_0003,business_report,2,,text,4466,20. 충당부채 (연결)....................................
3,000990_2025_business_report_0004,business_report,2,,text,1993,3. 중요한 회계추정 및 가정.................................
4,000990_2025_business_report_0005,business_report,3,,text,4123,19. 기타유동부채 (별도)..................................
5,000990_2025_business_report_0006,business_report,3,,text,2505,2. 내부통제에 관한 사항...................................
6,000990_2025_business_report_0007,business_report,4,,text,548,3. 타법인출자 현황(상세)..................................
7,000990_2025_business_report_0008,business_report,5,,text,324,사 업 보 고 서 (제 72 기) 2024년 01월 01일 부터 사업연도 2024년...
8,000990_2025_business_report_0009,business_report,6,,text,44,【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 2
9,000990_2025_business_report_0010,business_report,7,,text,29,전자공시시스템 dart.fss.or.kr Page 3


In [12]:
result = vectorstore.get(where={"doc_type": "sustainability_report"})
print(len(result["ids"]), "개 청크 확인됨")

275 개 청크 확인됨
